In [ ]:
# CELL 1: Setup and Mount
from google.colab import drive
import os

# 1. Mount Google Drive to save weights safely
drive.mount('/content/drive')

# 2. Create checkpoint directories on Drive
os.makedirs('/content/drive/MyDrive/FYP_PROJECT/FYP_Checkpoints/AdaLOLIE', exist_ok=True)
os.makedirs('/content/drive/MyDrive/FYP_PROJECT/FYP_Checkpoints/YOLO_Runs', exist_ok=True)

Mounted at /content/drive


In [ ]:
# CELL 2: Install required libraries
!pip install -q ultralytics comet_ml piq optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 90.6 MB/s eta 0:00:00


In [ ]:
# CELL 3: Unzip DsDPM_YOLO_Lightweight Datasets
print("📦 Unzipping DsDPM_YOLO_Lightweight to high-speed Colab storage...")
!unzip -q "/content/drive/MyDrive/FYP_PROJECT/Data/DsDPM_YOLO_Lightweight.zip" -d "/content/"

print("✅ Data Extraction Complete!")

📦 Unzipping DsDPM_YOLO_Lightweight to high-speed Colab storage...
✅ Data Extraction Complete!


In [ ]:
# CELL 4: Unzip MiningMix_Unified Datasets
print("📦 Unzipping MiningMix_Unified to high-speed Colab storage...")
!unzip -q "/content/drive/MyDrive/FYP_PROJECT/Data/MiningMix_Unified.zip" -d "/content/"

print("✅ Data Extraction Complete!")

📦 Unzipping MiningMix_Unified to high-speed Colab storage...
✅ Data Extraction Complete!


### Experiment 4: Combined Balancing (Focal Loss + Custom Class Penalties + Dynamic Image Sampling)

In [ ]:
import comet_ml

import os
import glob
import random
from collections import defaultdict
from ultralytics import YOLO

# Initialize Comet globally so YOLOv8 automatically detects it
os.environ["COMET_API_KEY"] = "BDA8b7jJyTFvOSAbOQnIOIOpf"
comet_ml.login(project_name="YOLO-Mining-Safety")

RUN_NAME = "exp4_custom_sampling"
CHECKPOINT_PATH = f"/content/drive/MyDrive/FYP_PROJECT/FYP_Checkpoints/YOLO_Runs/{RUN_NAME}/weights/last.pt"

LABEL_DIR = "/content/DsDPM_YOLO_Lightweight/labels/train"
class_to_images = defaultdict(list)

print("⚖️ Calculating data-driven class distribution...")
for label_file in glob.glob(os.path.join(LABEL_DIR, "*.txt")):
    with open(label_file) as f:
        classes = {int(line.split()[0]) for line in f if line.strip()}

    img_path = label_file.replace("labels", "images").replace(".txt", ".jpg")
    for c in classes:
        class_to_images[c].append(img_path)

# --- PURE DATA-DRIVEN OVERSAMPLING ---
# 1. Find the size of the majority class dynamically
max_class_size = max(len(imgs) for imgs in class_to_images.values())
print(f"📈 Majority class has {max_class_size} images. Balancing dataset...")

balanced_images = []
# 2. Oversample all classes to mathematically match the majority
for c, imgs in class_to_images.items():
    # random.choices allows duplicate sampling of rare images
    balanced_images.extend(random.choices(imgs, k=max_class_size))

# 3. Shuffle the giant list so YOLO doesn't train on the same class 10,000 times in a row
random.shuffle(balanced_images)

with open("/content/DsDPM_YOLO_Lightweight/balanced_train.txt", "w") as f:
    f.write("\n".join(balanced_images))

print(f"✅ Generated dynamic text dataloader with {len(balanced_images)} total images!")

# --- YAML GENERATION ---
yaml_content = """
path: /content/DsDPM_YOLO_Lightweight
train: balanced_train.txt
val: images/val
names:
  0: coal_miner
  1: drill_pipe
  2: drill_rig
  3: interaction_between_miner_and_drill_pipe
  4: mining_helmet
"""
with open("/content/DsDPM_YOLO_Lightweight/balanced_data.yaml", "w") as f:
    f.write(yaml_content.strip())

# --- HARDWARE-OPTIMIZED TRAINING LOOP ---
if os.path.exists(CHECKPOINT_PATH):
    print(f"🔄 Checkpoint found for {RUN_NAME}! Resuming training...")
    model = YOLO(CHECKPOINT_PATH)
    model.train(resume=True)
else:
    print(f"🆕 No checkpoint found. Starting {RUN_NAME} from scratch...")
    model = YOLO("yolov8n.pt")
    model.train(
      data="/content/DsDPM_YOLO_Lightweight/balanced_data.yaml",
      imgsz=640,
      epochs=100,
      batch=32,
      fraction=0.20,
      workers=8,
      cache=True,
      save_period=10,
      patience=20,
      optimizer='AdamW',
      cos_lr=True,
      project="/content/drive/MyDrive/FYP_PROJECT/FYP_Checkpoints/YOLO_Runs",
      name=RUN_NAME
    )

# 3. TELL COMET THE RUN IS OVER
print("✅ Training complete! Syncing final weights to Comet...")
comet_ml.end()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
⚖️ Calculating data-driven class distribution...
📈 Majority class has 62308 images. Balancing dataset...
✅ Generated dynamic text dataloader with 124616 total images!
🔄 Checkpoint found for exp4_custom_sampling! Resuming training...
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/DsDPM_YOLO_Lightweight/balanced_data.yaml, degrees=0.0, deterministic=True, devic

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/thaveesha-sonnadara/yolo-mining-safety/ec4d7bba375e4b7abe26cc25353e99a9

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.



                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : copper_peach_7714
COMET INFO:     url                   : https://www.comet.com/thaveesha-sonnadara/yolo-mining-safety/ec4d7bba375e4b7abe26cc25353e99a9
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [3112]               : (58785.53125, 356107.4375)
COMET INFO:     lr/pg0 [41]               : (0.0009559511573409196, 0.0062810149414660316)
COMET INFO:     lr/pg1 [41]               : (0.0009559511573409196, 0.0062810149414660316)
COMET INFO:     lr/pg2 [41]               : (0.0009559511573409196, 0.0062810149414660316)
COMET INFO:     metrics/mAP50(B) [41]     : (0.26418, 0.26735)
COMET INFO:     metrics/mAP50-95(B) [41]  : (0.2032, 0.20

✅ Training complete! Syncing final weights to Comet...


### AdaLOLIE Custom Training

In [ ]:
# AdaLOLIE Custom Training
import os
import comet_ml

# Ensure your API key is set so you don't get locked out of your dashboard!
# os.environ["COMET_API_KEY"] = "YOUR_API_KEY_HERE"

# Import ONLY the wrapper (it handles the model and loss internally)
from train import TrainScript

# Initialize Comet
experiment = comet_ml.start(project_name="AdaLOLIE-Mining-Safety")

# Initialize the trainer wrapper
trainer = TrainScript(experiment)

# --- OVERRIDE PATHS FOR COLAB ---
# Tell the script to look at the fast local storage
trainer.TRAIN_PATH = "/content/MiningMix_Unified/train"
trainer.VAL_PATH = "/content/MiningMix_Unified/val"
trainer.TEST_PATH = "/content/MiningMix_Unified/test"
trainer.SAVE_DIR = "/content/drive/MyDrive/FYP_PROJECT/FYP_Checkpoints/AdaLOLIE"

print("🚀 Starting AdaLOLIE Training Pipeline...")
trainer.train()

# Securely upload final weights
experiment.end()